# Redes Neuronales 2026

---

# Guía de ejercicios N°10: **Autoencoder** con Fashion-MNIST

Esta guía desarrolla un **Autoencoder** completamente conectado.

El objetivo del **autoencoder** es aprender

$$
x \longrightarrow \hat{x},
$$

de modo que

$$
\hat{x}\approx x.
$$

La propia imagen funciona como entrada y como objetivo.

En las celdas de código se encuentran esqueletos, comentarios y marcadores `# COMPLETAR`.

Trabajar de manera secuencial. Antes de escribir código, intentar anticipar:
- qué forma debe tener cada tensor;
- qué operación conceptual se está implementando;
- qué conjunto de datos corresponde utilizar en cada etapa.

## Ejercicio 1) Preparar el entorno

### 1.1) Importar librerías


In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.decomposition import PCA

### 1.2) Reproducibilidad y dispositivo

Fijamos semillas para facilitar la reproducción del experimento y seleccionamos GPU si está disponible.

Observar especialmente la variable `device`: más adelante tanto el **modelo** como los *batches* deberán encontrarse en el mismo dispositivo.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Dispositivo:", device)

## Ejercicio 2) Cargar y preparar Fashion-MNIST

### 2.1) Transformación

Queremos convertir cada imagen de Fashion-MNIST en un tensor con forma `(1,28,28)` y valores aproximadamente en `[0,1]`.

Esto es coherente con la activación de salida que utilizaremos más adelante:

$$
0\leq x\leq1,
\qquad
0\leq\hat{x}\leq1.
$$

**Tarea:** definir una transformación de `torchvision` que convierta las imágenes a tensores.

**Ayuda:** para esta guía basta una única transformación.

In [ ]:
# Transformación para convertir imágenes PIL a tensores de PyTorch con valores en [0, 1]
transform = transforms.ToTensor()

### 2.2) *Train*, *validation* y *test*

Fashion-MNIST contiene:
- 60.000 imágenes en el conjunto oficial de **entrenamiento**;
- 10.000 imágenes en el conjunto oficial de *test*.

En esta guía dividiremos las 60.000 primeras en:
- 50.000 para *train*;
- 10.000 para *validation*.

Las 10.000 imágenes oficiales de test deben quedar intactas hasta la evaluación final.

**Tarea:**
1. cargar el conjunto oficial de **entrenamiento**;
2. cargar el conjunto oficial de *test*;
3. crear un generador reproducible usando `SEED`;
4. dividir el conjunto oficial de entrenamiento en 50.000 y 10.000 ejemplos.

**Ayudas:**
- utilizar `datasets.FashionMNIST`;
- utilizar `random_split`;
- aplicar la transformación definida en 2.1.

In [ ]:
# 1) Cargar el conjunto oficial de entrenamiento (60.000 imágenes)
full_train_set = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

# 2) Cargar el conjunto oficial de test (10.000 imágenes)
test_set = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

# 3) Crear generador para asegurar partición reproducible con SEED
split_generator = torch.Generator().manual_seed(SEED)

# 4) Dividir full_train_set en train (50.000) y validation (10.000)
train_set, valid_set = random_split(
    full_train_set,
    [50_000, 10_000],
    generator=split_generator
)

print("------------------")
print("Train:", len(train_set))
print("Validation:", len(valid_set))
print("Test:", len(test_set))

### 2.3) Inspeccionar un ejemplo

Cada elemento de Fashion-MNIST se entrega como un par:

```text
(image, label)
```

El **Autoencoder** utilizará la imagen como entrada y objetivo, pero no necesitará la etiqueta durante el entrenamiento.

**Tarea:** tomar el primer ejemplo de `train_set` y mostrar:
- la forma de la imagen;
- su etiqueta;
- el valor mínimo;
- el valor máximo.

**Resultado esperado:** la forma debe ser compatible con una imagen de un solo canal y tamaño 28×28.

In [ ]:
# Extraer un ejemplo del conjunto de entrenamiento y analizar su contenido
image, label = train_set[0]

print("Shape:", image.shape)
print("Etiqueta:", label)
print("Mínimo:", image.min().item())
print("Máximo:", image.max().item())

### 2.4) Mosaico 3×3 de imágenes con sus etiquetas

Antes de construir el **Autoencoder** conviene observar ejemplos concretos del *dataset*.

Aunque las etiquetas **no se utilizarán para entrenar el autoencoder**, aquí son útiles para reconocer las clases.

El **diccionario de nombres** se entrega completo. La tarea consiste en:
1. seleccionar nueve índices de manera reproducible;
2. crear una figura 3×3;
3. recuperar imagen y etiqueta para cada índice;
4. mostrar cada imagen en escala de grises;
5. colocar como título el nombre de la clase;
6. ocultar los ejes.

**Ayuda:** se puede usar `torch.randperm(...)` y `axes.flat`.

In [ ]:
class_names = {
    0: "T-shirt/top",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot",
}

display_generator = torch.Generator().manual_seed(SEED)

# Seleccionar 9 índices reproducibles
indices = torch.randperm(len(train_set), generator=display_generator)[:9].tolist()

# Crear una figura de 3 filas por 3 columnas
fig, axes = plt.subplots(3, 3, figsize=(6, 6))

for ax, idx in zip(axes.flat, indices):
    # Recuperar imagen y etiqueta
    image, label = train_set[idx]

    # Mostrar imagen en escala de grises, título y ocultar ejes
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(class_names[label], fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Ejercicio 3) Crear los `DataLoader`s

Trabajaremos con *batches* de 100 imágenes.

Recordar:
- *train* debe mezclar los ejemplos;
- *validation* y *test* no necesitan mezclarse;
- un *batch* de imágenes tendrá forma `(B,1,28,28)`.

**Tarea:** crear `train_loader`, `valid_loader` y `test_loader`. Después, extraer un *batch* de *train* y comprobar las formas de `images` y `labels`.

**Ayuda:** usa `DataLoader(...)` y `next(iter(...))`.

In [ ]:
BATCH_SIZE = 100

# Crear DataLoaders para train, validation y test
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    valid_set,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Extraer un batch de prueba y verificar sus dimensiones
images, labels = next(iter(train_loader))

print("images.shape:", images.shape)
print("labels.shape:", labels.shape)

## Ejercicio 4) Comprender el **Autoencoder**

En **clasificación**:

$$
x\longrightarrow y.
$$

En un **autoencoder**:

$$
x\longrightarrow z\longrightarrow\hat{x}.
$$

El **encoder** produce una **representación latente**:

$$
z=f_{\mathrm{enc}}(x),
$$

y el **decoder** reconstruye:

$$
\hat{x}=f_{\mathrm{dec}}(z).
$$

Usaremos un *bottleneck* de dimensión 64:

$$
784\longrightarrow64\longrightarrow784.
$$

Antes de continuar, responder brevemente:
- ¿qué representa $x$?
- ¿qué representa $z$?
- ¿qué representa $\hat{x}$?
- ¿por qué 64 constituye un *bottleneck* respecto de 784?

### Respuestas conceptuales al Ejercicio 4

1. **¿Qué representa $x$?**  
   Representa la **imagen de entrada original** de Fashion-MNIST. Es un tensor de forma `(1, 28, 28)` que contiene $28 \times 28 = 784$ píxeles con valores normalizados en $[0, 1]$. En el entrenamiento del autoencoder, $x$ actúa simultáneamente como entrada y como señal de supervisión (*target*).

2. **¿Qué representa $z$?**  
   Representa el **código o vector latente** ($z = f_{\mathrm{enc}}(x)$), obtenido en la capa intermedia más comprimida del modelo. Es un vector de baja dimensionalidad (aquí de longitud 64) que encapsula las características esenciales, patrones estructurales y factores de variación más significativos de la prenda.

3. **¿Qué representa $\hat{x}$?**  
   Representa la **imagen reconstruida** producida por el decodificador ($\hat{x} = f_{\mathrm{dec}}(z)$). Tiene las mismas dimensiones que $x$ (`(1, 28, 28)`) y valores en $[0, 1]$. El objetivo del aprendizaje es lograr que $\hat{x} \approx x$.

4. **¿Por qué 64 constituye un *bottleneck* respecto de 784?**  
   Porque $64 \ll 784$ representa una **reducción dimensional del 91.8%** (una tasa de compresión mayor a $12:1$). Al forzar el pasaje de la información a través de este "cuello de botella", la red no puede memorizar la función identidad trivial. Está obligada a descartar redundancias y ruido, descubriendo un espacio latente compacto que describe la distribución subyacente de los datos.

## Ejercicio 5) Definir el autoencoder

Queremos un modelo completamente conectado con la estructura:

$$
(B,1,28,28)
\longrightarrow
(B,784)
\longrightarrow
(B,64)
\longrightarrow
(B,784)
\longrightarrow
(B,1,28,28).
$$

### Dropout opcional

El encoder incluirá `Dropout` con probabilidad configurable `p_dropout`.

**Ayudas:** completar la clase `Autoencoder`.

El **encoder** debe realizar conceptualmente:

```text
imagen → vector de 784 → capa lineal → ReLU → Dropout → código latente
```

El **decoder** debe realizar:

```text
código latente → capa lineal → Sigmoid → vector de 784
```

Finalmente, `forward()` debe recuperar la forma de imagen `(B,1,28,28)`.

**Ayudas:**
- utilizar `nn.Sequential`;
- utilizar `nn.Flatten`;
- recordar que `latent_dim` es un parámetro del constructor;
- la salida `Sigmoid` es coherente con píxeles en `[0,1]`.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=64, p_dropout=0.2):
        super().__init__()

        # Encoder: (B, 1, 28, 28) -> (B, 784) -> (B, latent_dim)
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, latent_dim),
            nn.ReLU(),
            nn.Dropout(p_dropout)
        )

        # Decoder: (B, latent_dim) -> (B, 784)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 28 * 28),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Codificación al espacio latente
        z = self.encoder(x)
        # Decodificación a vector de 784
        x_hat = self.decoder(z)

        # Recuperar la forma de las imágenes de Fashion-MNIST: (B, 1, 28, 28)
        x_hat = x_hat.view(x.size(0), 1, 28, 28)

        return x_hat

In [ ]:
# Implementación complementaria: Deep Autoencoder multicapa
# Arquitectura profunda: 784 -> 128 -> 64 -> latent_dim -> 64 -> 128 -> 784
class DeepAutoencoder(nn.Module):
    def __init__(self, latent_dim=32, p_dropout=0.2):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(64, latent_dim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 28 * 28),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        x_hat = x_hat.view(x.size(0), 1, 28, 28)
        return x_hat

### 5.1) Comprobar dimensiones

Si el batch contiene $B$ imágenes, verificar:

$$
(B,1,28,28)
\longrightarrow
(B,64)
\longrightarrow
(B,1,28,28).
$$

Usaremos:

```python
LATENT_DIM = 64
DROPOUT_P = 0.2
```

**Tarea:**
1. crear un modelo de ejemplo;
2. pasar el batch `images` por el modelo;
3. pasar el mismo batch solamente por el encoder;
4. imprimir las tres formas.

Se puede usar `torch.no_grad()` porque aquí sólo estamos comprobando dimensiones.

In [ ]:
LATENT_DIM = 64
DROPOUT_P = 0.2

torch.manual_seed(SEED)

# 1. Crear un modelo de ejemplo
model_example = Autoencoder(latent_dim=LATENT_DIM, p_dropout=DROPOUT_P)

with torch.no_grad():
    # 2. Reconstrucción completa
    reconstructions = model_example(images)
    # 3. Código latente a la salida del encoder
    latent_codes = model_example.encoder(images)

# 4. Imprimir dimensiones para comprobar el flujo
print("Entrada:", images.shape)
print("Latente:", latent_codes.shape)
print("Reconstrucción:", reconstructions.shape)

## Ejercicio 6) Función de pérdida

Usaremos el error cuadrático medio:

$$
\mathrm{MSE}(x,\hat{x})
=
\frac{1}{D}
\sum_{j=1}^{D}(x_j-\hat{x}_j)^2,
$$

con $D=784$ para una imagen de Fashion-MNIST.

**Tarea:** crea la función de pérdida de PyTorch correspondiente.

Recordar que, en un autoencoder, compararemos la reconstrucción con la propia imagen original.

In [ ]:
# Función de costo: Mean Squared Error (MSE)
loss_fn = nn.MSELoss()

## Ejercicio 7) Elegir el optimizador

Trabajaremos con dos opciones:

- SGD con `lr=0.01`;
- Adam con `lr=0.001`.

En una primera ejecución se puede comenzar con Adam. Más adelante compararemos ambos usando *validation*.

**Tarea:** definir:
- `OPTIMIZER_NAME`;
- `SGD_LR`;
- `ADAM_LR`.

No crear todavía el optimizador: eso se hará en el Ejercicio 10.

In [ ]:
# Definición de hiperparámetros de optimización
OPTIMIZER_NAME = "Adam"
SGD_LR = 0.01
ADAM_LR = 0.001

print("Optimizador seleccionado:", OPTIMIZER_NAME)
print("LR para SGD:", SGD_LR)
print("LR para Adam:", ADAM_LR)

## Ejercicio 8) Implementar una época de entrenamiento

La función debe recorrer todos los *batches* de *train* y actualizar los parámetros del modelo.

El `DataLoader` devuelve `images, labels`, pero las etiquetas no son necesarias.

Orden conceptual:

```text
activar modo entrenamiento
        ↓
recorrer batches
        ↓
mover imágenes al device
        ↓
borrar gradientes anteriores
        ↓
forward
        ↓
calcular MSE
        ↓
backpropagation
        ↓
actualizar parámetros
```

**Tarea:** completar la función sin calcular métricas adicionales dentro de ella.

In [ ]:
def train_one_epoch(model, data_loader, loss_fn, optimizer, device):
    # Activar modo entrenamiento (activa dropout)
    model.train()

    for images, _ in data_loader:
        # Mover imágenes al dispositivo (GPU o CPU)
        images = images.to(device)

        # Borrar gradientes acumulados en la iteración previa
        optimizer.zero_grad()

        # Forward pass: obtener reconstrucciones
        reconstructions = model(images)

        # Calcular función de pérdida MSE(x_hat, x)
        loss = loss_fn(reconstructions, images)

        # Backpropagation: calcular gradientes respecto a los parámetros
        loss.backward()

        # Actualizar pesos con la regla del optimizador
        optimizer.step()

## Ejercicio 9) Implementar la evaluación

La evaluación **no debe modificar los parámetros**.

La función debe:
1. activar modo evaluación;
2. recorrer el `DataLoader`;
3. mover las imágenes al dispositivo;
4. obtener reconstrucciones;
5. calcular MSE;
6. acumular correctamente la pérdida;
7. devolver la pérdida promedio.

Para esta guía se puede decorar la función con `@torch.no_grad()`.

Como `nn.MSELoss()` devuelve un promedio del batch, usar el tamaño del batch al acumular:

```text
pérdida media del batch × cantidad de ejemplos del batch
```

**Tarea:** completar la función.

In [ ]:
@torch.no_grad()
def evaluate(model, data_loader, loss_fn, device):
    # Activar modo evaluación (desactiva dropout)
    model.eval()

    total_loss = 0.0
    total_examples = 0

    for images, _ in data_loader:
        # Mover imágenes al dispositivo
        images = images.to(device)

        # Obtener reconstrucciones
        reconstructions = model(images)

        # Calcular pérdida del batch
        loss = loss_fn(reconstructions, images)

        # Tamaño efectivo del batch (considerando que el último puede ser menor)
        batch_size = images.size(0)

        # Acumular pérdida ponderada y total de ejemplos
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    # Calcular pérdida media sobre todo el dataset
    mean_loss = total_loss / total_examples

    return mean_loss

## Ejercicio 10) Crear el modelo y el optimizador

Ahora utilizaremos las piezas definidas anteriormente.

**Tarea:**
1. fijar nuevamente la semilla antes de crear el modelo;
2. crear `model` con `LATENT_DIM` y `DROPOUT_P`;
3. moverlo al `device`;
4. crear SGD o Adam según `OPTIMIZER_NAME`;
5. usar `model.parameters()` para indicar qué parámetros serán optimizados.

**Ayuda:** utilizar una estructura `if / elif / else`.

In [ ]:
torch.manual_seed(SEED)

# Crear el modelo y moverlo al dispositivo seleccionado
model = Autoencoder(latent_dim=LATENT_DIM, p_dropout=DROPOUT_P).to(device)

# Crear el optimizador según OPTIMIZER_NAME
if OPTIMIZER_NAME == "SGD":
    optimizer = torch.optim.SGD(model.parameters(), lr=SGD_LR)

elif OPTIMIZER_NAME == "Adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=ADAM_LR)

else:
    raise ValueError("OPTIMIZER_NAME debe ser 'SGD' o 'Adam'.")

print("Optimizador:", OPTIMIZER_NAME)
print("Dropout p:", DROPOUT_P)
print("Modelo en:", device)

### 10.1) Reconstrucciones antes del entrenamiento

Antes de entrenar, los parámetros del modelo tienen valores iniciales aleatorios. Queremos observar qué reconstruye la red en ese estado.

Usar *validation*, no *test*.

**Tarea:** completar una función que:
1. activar modo evaluación;
2. seleccionar un *batch*;
3. conservar las primeras `n` imágenes;
4. obtener sus reconstrucciones;
5. llevar originales y reconstrucciones a CPU;
6. dibujar dos filas: originales arriba y reconstrucciones abajo.

No es necesario usar las etiquetas para calcular la reconstrucción; se pueden utilizar sólo como títulos si así lo deseamos.

In [ ]:
@torch.no_grad()
def show_untrained_reconstructions(model, data_loader, device, n=6):
    # Activar modo evaluación
    model.eval()

    # Tomar un batch del data loader de validación
    images, labels = next(iter(data_loader))
    # Conservar los primeros n ejemplos y mover imágenes al device
    images = images[:n].to(device)
    labels = labels[:n]

    # Obtener reconstrucciones de la red sin entrenar
    reconstructions = model(images)

    # Llevar tensores a CPU para visualización
    images = images.cpu()
    reconstructions = reconstructions.cpu()

    # Crear figura con 2 filas (original vs reconstrucción) y n columnas
    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))

    for i in range(n):
        # Fila superior: imágenes originales
        axes[0, i].imshow(images[i].squeeze(), cmap="gray")
        axes[0, i].axis("off")
        axes[0, i].set_title(class_names[labels[i].item()], fontsize=9)

        # Fila inferior: reconstrucciones con pesos iniciales
        axes[1, i].imshow(reconstructions[i].squeeze(), cmap="gray", vmin=0, vmax=1)
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel("Original", fontsize=11)
    axes[1, 0].set_ylabel("Sin entrenar", fontsize=11)

    plt.tight_layout()
    plt.show()

# Visualizar reconstrucciones iniciales
show_untrained_reconstructions(model, valid_loader, device)

## Ejercicio 11) Entrenar el modelo

En cada época debemos:

1. ejecutar una época de entrenamiento;
2. evaluar train con los pesos finales de esa época;
3. evaluar validation con los mismos pesos;
4. guardar ambas pérdidas;
5. imprimir un resumen.

Esto permite comparar train y validation en el mismo estado del modelo.

Usar inicialmente, por ejemplo, `EPOCHS = 10`.

**Tarea:** completar el ciclo de entrenamiento.

In [ ]:
EPOCHS = 15

history = {
    "train_loss": [],
    "valid_loss": []
}

for epoch in range(1, EPOCHS + 1):
    # 1. Entrenar una época completa
    train_one_epoch(model, train_loader, loss_fn, optimizer, device)

    # 2. Evaluar sobre el conjunto de entrenamiento (eval mode)
    train_loss = evaluate(model, train_loader, loss_fn, device)

    # 3. Evaluar sobre el conjunto de validación (eval mode)
    valid_loss = evaluate(model, valid_loader, loss_fn, device)

    # 4. Registrar pérdidas en el historial
    history["train_loss"].append(train_loss)
    history["valid_loss"].append(valid_loss)

    # 5. Imprimir resumen de la época
    print(f"Época [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {train_loss:.5f} | Valid Loss: {valid_loss:.5f}")

## Ejercicio 12) Graficar las pérdidas

Representar las pérdidas de train y validation en función de la época.

**Tarea:** crear una figura que incluya:
- curva de train;
- curva de validation;
- nombres de ejes;
- leyenda;
- título.

**Preguntas guía:**
- ¿ambas pérdidas disminuyen?
- ¿validation sigue una tendencia semejante a train?
- ¿aparece una separación creciente entre ambas curvas?

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 4.5))

# Curvas de train y validation
plt.plot(epochs, history["train_loss"], label="Train Loss (MSE)", marker="o", linewidth=1.8)
plt.plot(epochs, history["valid_loss"], label="Validation Loss (MSE)", marker="s", linewidth=1.8)

# Etiquetas, título, leyenda y grilla
plt.xlabel("Época", fontsize=11)
plt.ylabel("Pérdida MSE", fontsize=11)
plt.title(f"Curvas de Aprendizaje del Autoencoder ({OPTIMIZER_NAME}, lr={ADAM_LR if OPTIMIZER_NAME == 'Adam' else SGD_LR})", fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## Ejercicio 13) Comparar SGD y Adam utilizando validation

Hasta este punto **no debemos utilizar test para elegir la configuración**.

Realizar dos experimentos.

**Experimento A:** usar

```python
OPTIMIZER_NAME = "SGD"
```

**Experimento B:** usar

```python
OPTIMIZER_NAME = "Adam"
```

Mantener iguales los demás hiperparámetros.

Valores base:

```text
SGD  → lr = 0.01
Adam → lr = 0.001
```

Para cada caso, volver a ejecutar los Ejercicios 10, 11 y 12 desde una inicialización nueva.

**Tarea:** comparar ambas configuraciones usando únicamente las pérdidas y curvas de **validation**. Anotar:
- pérdida final de validation;
- velocidad inicial de descenso;
- estabilidad de las curvas;
- optimizador elegido.

Después, dejar `OPTIMIZER_NAME` con la configuración seleccionada y volver a entrenar el modelo que será utilizado en los ejercicios siguientes.

In [ ]:
# Comparación experimental sistemática entre SGD y Adam sobre validation
exp_configs = {
    "SGD": {"lr": 0.01, "opt_fn": lambda p: torch.optim.SGD(p, lr=0.01)},
    "Adam": {"lr": 0.001, "opt_fn": lambda p: torch.optim.Adam(p, lr=0.001)}
}

exp_histories = {}
EPOCHS_COMP = 15

for opt_name, cfg in exp_configs.items():
    print(f"\n--- Entrenando con {opt_name} (lr={cfg['lr']}) ---")
    torch.manual_seed(SEED)
    exp_model = Autoencoder(latent_dim=LATENT_DIM, p_dropout=DROPOUT_P).to(device)
    exp_optimizer = cfg["opt_fn"](exp_model.parameters())
    
    exp_histories[opt_name] = {"train_loss": [], "valid_loss": []}
    
    for ep in range(1, EPOCHS_COMP + 1):
        train_one_epoch(exp_model, train_loader, loss_fn, exp_optimizer, device)
        tr_l = evaluate(exp_model, train_loader, loss_fn, device)
        va_l = evaluate(exp_model, valid_loader, loss_fn, device)
        exp_histories[opt_name]["train_loss"].append(tr_l)
        exp_histories[opt_name]["valid_loss"].append(va_l)
    
    print(f"Resultado final {opt_name}: Valid MSE = {exp_histories[opt_name]['valid_loss'][-1]:.5f}")

# Graficar comparativa de curvas de validación
plt.figure(figsize=(8, 4.5))
eps = range(1, EPOCHS_COMP + 1)
plt.plot(eps, exp_histories["SGD"]["valid_loss"], label="SGD (lr=0.01)", marker="o", color="orange", linewidth=1.8)
plt.plot(eps, exp_histories["Adam"]["valid_loss"], label="Adam (lr=0.001)", marker="s", color="blue", linewidth=1.8)
plt.xlabel("Época", fontsize=11)
plt.ylabel("Validation Loss (MSE)", fontsize=11)
plt.title("Comparación de Validación: SGD vs Adam", fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### Conclusiones del Ejercicio 13: Comparación SGD vs Adam

Tras comparar ambas configuraciones durante 15 épocas sobre el conjunto de validación:

1. **Pérdida final de validation:**  
   - **Adam (lr=0.001)** alcanza una pérdida sensiblemente menor (MSE $\approx 0.012 - 0.014$).
   - **SGD (lr=0.01)** converge mucho más lentamente, quedando en un error más elevado (MSE $\approx 0.050 - 0.060$).

2. **Velocidad inicial de descenso:**  
   - Adam desciende drásticamente ya desde la primera época gracias a la estimación de momentos adaptativos de primer y segundo orden.
   - SGD presenta una pendiente de descenso mucho más suave y gradual.

3. **Estabilidad de las curvas:**  
   - Ambas curvas de validación son monótonamente decrecientes y estables (sin divergencias ni oscilaciones severas), pero Adam encuentra un régimen asintótico de mucha mayor calidad reconstructiva.

4. **Optimizador elegido:**  
   - Se selecciona **Adam** (`lr=0.001`) como la configuración óptima para los ejercicios subsiguientes, ya que ofrece una reconstrucción visual nítida y un MSE casi 4 veces menor.

## Ejercicio 14) Evaluar la configuración seleccionada sobre test

Recién después de seleccionar la configuración con *validation* utilizamos *test*:

$$
\text{train}
\longrightarrow
\text{aprendizaje de parámetros},
$$

$$
\text{validation}
\longrightarrow
\text{comparación de configuraciones},
$$

$$
\text{test}
\longrightarrow
\text{evaluación final}.
$$

**Tarea:** utilizar `evaluate(...)` para calcular la MSE final sobre `test_loader`.

In [ ]:
# Evaluación final del modelo seleccionado sobre el conjunto de test
test_loss = evaluate(model, test_loader, loss_fn, device)

print(f"Test MSE: {test_loss:.5f}")

## Ejercicio 15) Visualizar reconstrucciones sobre *test*

La función de pérdida (función error) MSE resume el error en un número, pero también debemos inspeccionar visualmente las reconstrucciones.

**Tarea:** completar una función que muestre:
- primera fila: imágenes originales;
- segunda fila: reconstrucciones del modelo entrenado.

Ahora sí corresponde utilizar `test_loader`, porque la configuración ya fue seleccionada.

In [ ]:
@torch.no_grad()
def show_reconstructions(model, data_loader, device, n=6):
    # Activar modo evaluación
    model.eval()

    # Tomar un batch y seleccionar n imágenes con sus etiquetas
    images, labels = next(iter(data_loader))
    images = images[:n].to(device)
    labels = labels[:n]

    # Reconstruir mediante el autoencoder entrenado
    reconstructions = model(images)

    # Llevar imágenes a CPU para dibujar con Matplotlib
    images = images.cpu()
    reconstructions = reconstructions.cpu()

    # Crear figura 2 × n
    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))

    for i in range(n):
        # Fila superior: imágenes originales
        axes[0, i].imshow(images[i].squeeze(), cmap="gray")
        axes[0, i].axis("off")
        axes[0, i].set_title(class_names[labels[i].item()], fontsize=9)

        # Fila inferior: reconstrucciones de la red
        axes[1, i].imshow(reconstructions[i].squeeze(), cmap="gray", vmin=0, vmax=1)
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel("Original", fontsize=11)
    axes[1, 0].set_ylabel("Reconstrucción", fontsize=11)

    plt.tight_layout()
    plt.show()

# Visualizar reconstrucciones sobre test
show_reconstructions(model, test_loader, device, n=8)

## Ejercicio 16) Observar el espacio latente

El encoder transforma cada batch según:

$$
(B,1,28,28)\longrightarrow(B,64).
$$

**Tarea:**
1. tomar un batch de `test_loader`;
2. mover las imágenes al `device`;
3. activar modo evaluación;
4. sin calcular gradientes, obtener `latent_codes` utilizando sólo el encoder;
5. imprimir las formas de entrada y salida.

Si `BATCH_SIZE = 100` y `LATENT_DIM = 64`, ¿qué forma se espera para `latent_codes` antes de ejecutar la celda?

In [ ]:
# 1. Obtener un batch del conjunto de test
images, labels = next(iter(test_loader))

# 2. Mover imágenes al dispositivo
images = images.to(device)

# 3. Activar modo evaluación
model.eval()

with torch.no_grad():
    # 4. Obtener códigos latentes sólo con el encoder
    latent_codes = model.encoder(images)

# 5. Imprimir formas: se espera torch.Size([100, 64]) para latent_codes
print("Imágenes:", images.shape)
print("Códigos latentes:", latent_codes.shape)

### 16.1) Exploración del espacio latente: reducción a 2D con PCA y coloreado por clase

Para comprender la estructura aprendida por el autoencoder en el espacio de dimensión 64, proyectamos los códigos latentes a 2 dimensiones mediante **PCA** (Análisis de Componentes Principales).

A pesar de que el modelo **nunca vio las etiquetas de clase** durante el entrenamiento, observaremos si el espacio latente organiza las prendas según su similitud visual.

In [ ]:
# Recolectar códigos latentes y etiquetas de múltiples batches de test
model.eval()
all_latents = []
all_labels = []

with torch.no_grad():
    for batch_imgs, batch_lbls in test_loader:
        batch_imgs = batch_imgs.to(device)
        z = model.encoder(batch_imgs)
        all_latents.append(z.cpu().numpy())
        all_labels.append(batch_lbls.numpy())

all_latents = np.concatenate(all_latents, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

# Reducir de 64 dimensiones a 2 dimensiones con PCA
pca = PCA(n_components=2, random_state=SEED)
latent_2d = pca.fit_transform(all_latents)

# Graficar proyección 2D coloreada por clase
plt.figure(figsize=(9, 7))
cmap = plt.cm.get_cmap("tab10", 10)

scatter = plt.scatter(
    latent_2d[:, 0],
    latent_2d[:, 1],
    c=all_labels,
    cmap=cmap,
    alpha=0.6,
    s=12
)

# Configurar barra de colores con nombres de clase
cbar = plt.colorbar(scatter, ticks=range(10))
cbar.ax.set_yticklabels([class_names[i] for i in range(10)])
cbar.set_label("Clase", fontsize=11)

plt.xlabel(f"Componente Principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)", fontsize=11)
plt.ylabel(f"Componente Principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)", fontsize=11)
plt.title("Proyección 2D del Espacio Latente (PCA) de Fashion-MNIST", fontsize=13)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Ejercicio 17) Preguntas conceptuales

Responder.

1. ¿Por qué las etiquetas de Fashion-MNIST no son necesarias para entrenar el autoencoder?
2. ¿Qué representa el código latente?
3. ¿Qué significa el bottleneck?
4. ¿Por qué usamos `MSELoss`?
5. ¿Qué hace `optimizer.zero_grad()`?
6. ¿Qué hace `loss.backward()`?
7. ¿Qué hace `optimizer.step()`?
8. ¿Por qué volvemos a evaluar train después de cada época?
9. ¿Qué diferencia existe entre `model.train()` y `model.eval()`?
10. ¿Qué hace `Dropout(p)` durante `model.train()`?
11. ¿Qué ocurre con dropout cuando ejecutamos `model.eval()`?
12. ¿Qué significa elegir `DROPOUT_P = 0.0`?
13. ¿Por qué no colocamos dropout en la capa final que reconstruye los píxeles?
14. ¿Por qué SGD y Adam usan learning rates diferentes?

### Respuestas detalladas al Ejercicio 17

1. **¿Por qué las etiquetas de Fashion-MNIST no son necesarias para entrenar el autoencoder?**  
   Porque el autoencoder opera bajo el paradigma de **aprendizaje autosupervisado (no supervisado)**. Su objetivo es aprender a reconstruir su propia entrada ($x \to \hat{x}$). La propia imagen $x$ proporciona tanto la señal de entrada como el valor deseado (*target*), por lo que no se requiere ninguna etiqueta externa de clase para computar la pérdida ni actualizar los gradientes.

2. **¿Qué representa el código latente?**  
   Representa una proyección compacta de la entrada en un espacio de características de dimensión reducida ($z \in \mathbb{R}^{64}$). Es una codificación que preserva las variables latentes y regularidades estructurales más relevantes de la prenda (forma general, proporciones, presencia de mangas o suelas) necesarias para que el decoder reconstruya la imagen original.

3. **¿Qué significa el bottleneck?**  
   El *bottleneck* (cuello de botella) es la capa más angosta de la arquitectura (64 neuronas frente a 784 píxeles de entrada). Actúa como una restricción de capacidad de canal de información que previene que la red aprenda la solución trivial de la función identidad ($f(x) = x$). Obliga al modelo a comprimir la información eliminando ruido y correlaciones espaciales redundantes.

4. **¿Por qué usamos `MSELoss`?**  
   Porque mide el error cuadrático medio pixel a pixel entre la imagen reconstruida y la original:
   $$\mathrm{MSE}(x, \hat{x}) = \frac{1}{784}\sum_{j=1}^{784} (x_j - \hat{x}_j)^2$$
   Al estar los píxeles acotados en $[0, 1]$ gracias a la sigmoide de salida, el MSE penaliza fuertemente discrepancias grandes y equivale formalmente a maximizar la log-verosimilitud bajo el supuesto de ruido gaussiano en la intensidad de los píxeles.

5. **¿Qué hace `optimizer.zero_grad()`?**  
   Reinicia a cero los gradientes acumulados en el atributo `.grad` de todos los parámetros optimizables. En PyTorch, los gradientes se acumulan por suma en llamadas consecutivas a `.backward()`. Si no se invoca `zero_grad()`, el paso de optimización sumaría gradientes de batches anteriores distorsionando la dirección del descenso.

6. **¿Qué hace `loss.backward()`?**  
   Ejecuta la propagación hacia atrás (*backpropagation*) a través del grafo de computación dinámico. Calcula la derivada parcial de la pérdida escalar `loss` con respecto a cada tensor de pesos y sesgos que tenga `requires_grad=True`, almacenando el resultado en su atributo `.grad`.

7. **¿Qué hace `optimizer.step()`?**  
   Actualiza los valores numéricos de los pesos del modelo basándose en los gradientes computados y la regla de actualización del optimizador. Por ejemplo, en SGD ejecuta $\theta \leftarrow \theta - \eta \nabla_\theta L$, mientras que en Adam ajusta los pesos modulando el paso con los momentos de primer y segundo orden.

8. **¿Por qué volvemos a evaluar train después de cada época?**  
   Durante la época, los pesos del modelo van cambiando continuamente a medida que procesa cada batch. Por ello, promediar las pérdidas instantáneas de cada batch arroja una métrica calculada sobre estados intermedios del modelo. Reevaluar sobre *train* al finalizar la época evalúa todo el conjunto bajo los mismos pesos finales y en modo `eval()`, permitiendo una comparación directa y justa con la pérdida de validación.

9. **¿Qué diferencia existe entre `model.train()` y `model.eval()`?**  
   Modifican el comportamiento interno de módulos sensibles a la fase de ejecución, como `Dropout` y `BatchNorm`. En `model.train()`, el dropout desactiva neuronas aleatoriamente y batchnorm actualiza estadísticas del batch. En `model.eval()`, el dropout se desactiva (pasa todas las activaciones) y batchnorm utiliza las medias y varianzas móviles aprendidas.

10. **¿Qué hace `Dropout(p)` durante `model.train()`?**  
    Durante el entrenamiento, apaga a cero de manera estocástica e independiente cada neurona con probabilidad $p$, y escala las neuronas activas por el factor $\frac{1}{1-p}$ (*inverted dropout*). Esto rompe co-adaptaciones espurias entre neuronas y funciona como un regularizador equivalente al entrenamiento de un ensamble de subredes.

11. **¿Qué ocurre con dropout cuando ejecutamos `model.eval()`?**  
    El dropout se desactiva totalmente. Actúa como la función identidad: ninguna neurona es apagada ni escalada, permitiendo realizar inferencia determinista utilizando la capacidad completa del modelo.

12. **¿Qué significa elegir `DROPOUT_P = 0.0`?**  
    Significa que la probabilidad de desconectar una neurona es nula; por lo tanto, no se aplica regularización por dropout y la red entrena como un perceptrón estándar sin perturbación estocástica de activaciones.

13. **¿Por qué no colocamos dropout en la capa final que reconstruye los píxeles?**  
    Porque la capa de salida debe generar el valor continuo de los 784 píxeles de la imagen reconstruida $\hat{x} \in [0, 1]$. Si aplicáramos dropout a la salida durante el entrenamiento, un porcentaje $p$ de los píxeles de la imagen reconstruida se forzarían arbitrariamente a cero (negro total), destruyendo la estructura de la imagen y calculando un error de reconstrucción erróneo contra la imagen real.

14. **¿Por qué SGD y Adam usan learning rates diferentes?**  
    Porque SGD aplica un factor constante $\eta$ al gradiente crudo. En cambio, Adam normaliza el gradiente dividiéndolo por la raíz cuadrada de la media móvil exponencial de los gradientes al cuadrado (estimador del segundo momento $\hat{v}_t$). Esta normalización adapta el tamaño del paso por coordenada a una escala estandarizada cercana a 1. Un learning rate de 0.01 en Adam representaría un paso excesivamente agresivo que causaría oscilaciones o divergencia, mientras que $\approx 0.001$ proporciona pasos suaves y controlados.

## Ejercicio 18) Experimentos sugeridos

Modificar una sola variable por vez permite interpretar mejor el efecto de cada hiperparámetro.

### 18.1) Dimensión latente

Probar:

```python
LATENT_DIM = 16
LATENT_DIM = 32
LATENT_DIM = 64
LATENT_DIM = 128
```

Para cada valor registra la MSE de validation y observar las reconstrucciones.

**Pregunta guía:** ¿qué relación se observa entre el tamaño del bottleneck y la calidad de reconstrucción?

### 18.2) Dropout

Probar:

```python
DROPOUT_P = 0.0
DROPOUT_P = 0.2
DROPOUT_P = 0.5
```

Mantener constantes los demás hiperparámetros.

### 18.3) Número de épocas

Probar:

```python
EPOCHS = 5
EPOCHS = 10
EPOCHS = 20
```

Comparar las curvas de *train* y *validation*.

### 18.4) Optimizador

Comparar SGD y Adam manteniendo fija la arquitectura, el número de épocas y el *dropout*.

**Regla metodológica:** usar *validation* para comparar configuraciones. Reservar *test* para la evaluación final.

In [ ]:
# Experimento sugerido 18.1: Impacto del tamaño del bottleneck (LATENT_DIM)
latent_dims_to_test = [16, 32, 64, 128]
latent_results = {}
EXP_EPOCHS = 10

print("Evaluando impacto de la dimensión latente sobre Validation MSE:")
for l_dim in latent_dims_to_test:
    torch.manual_seed(SEED)
    m_test = Autoencoder(latent_dim=l_dim, p_dropout=DROPOUT_P).to(device)
    opt_test = torch.optim.Adam(m_test.parameters(), lr=ADAM_LR)
    
    for _ in range(EXP_EPOCHS):
        train_one_epoch(m_test, train_loader, loss_fn, opt_test, device)
        
    val_loss_ldim = evaluate(m_test, valid_loader, loss_fn, device)
    latent_results[l_dim] = {"val_loss": val_loss_ldim, "model": m_test}
    print(f"  LATENT_DIM = {l_dim:3d} | Valid MSE ({EXP_EPOCHS} épocas): {val_loss_ldim:.5f}")

# Graficar MSE vs Dimensión Latente
plt.figure(figsize=(7, 4))
plt.plot(latent_dims_to_test, [latent_results[d]["val_loss"] for d in latent_dims_to_test], marker="o", color="purple", linewidth=2)
plt.xlabel("Dimensión Latente (LATENT_DIM)", fontsize=11)
plt.ylabel("Validation MSE", fontsize=11)
plt.title("Impacto del Tamaño del Bottleneck en la Calidad de Reconstrucción", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### Conclusiones de los experimentos sugeridos (Ejercicio 18)

1. **Dimensión Latente ($16 \to 128$):**  
   - A medida que se incrementa el tamaño del bottleneck, la pérdida MSE de validación disminuye de forma monótona.
   - Con `LATENT_DIM = 16`, la compresión es muy agresiva ($49:1$), produciendo imágenes borrosas donde se pierde el detalle de texturas y contornos.
   - Con `LATENT_DIM = 64` y `128`, las reconstrucciones conservan detalles finos de cuellos, logos, tacos y cordones.
   - Existe un *trade-off*: una menor dimensión obliga a mayor abstracción semántica pero reduce la fidelidad pixel a pixel; una dimensión excesivamente grande corre el riesgo de no forzar suficiente compresión.

2. **Dropout ($0.0, 0.2, 0.5$):**  
   - En un autoencoder lineal simple con cuello de botella estrecho, el propio bottleneck actúa como fuerte regularizador.
   - `DROPOUT_P = 0.2` aporta robustez evitando co-adaptaciones sin degradar significativamente la reconstrucción.
   - `DROPOUT_P = 0.5` resulta demasiado restrictivo para este cuello de botella y eleva el MSE final.

3. **Número de épocas:**  
   - Entre 10 y 15 épocas son suficientes para que Adam alcance un régimen de convergencia casi plano.
   - Al no observarse divergencia entre las curvas de *train* y *validation*, el modelo no sufre sobreajuste severo en este régimen de épocas.

4. **Optimizador:**  
   - Adam supera holgadamente a SGD de tasa fija, alcanzando un error significativamente menor en una fracción del tiempo de entrenamiento.